# Notebook for reducing NIRCam WFSS Data

## The reduction steps in this notebook follow the procedure outlined in [Kashino+23](https://arxiv.org/abs/2211.08254)

### Note: This notebook is currently optimised for medium band images with very few dithers (GO Programs [2883](https://www.stsci.edu/jwst-program-info/program/?program=2883) and [3538](https://www.stsci.edu/jwst-program-info/program/?program=3538)) Background subtraction is therefore performed locally, per image. Observations with more dithers should instead build an empirical background.

This notebook is designed to work directly with [grizli](https://github.com/gbrammer/grizli) and therefore follows the same directory structure.
- Files downloaded from mast for reduction should be stored in './Raw'
- Outputted files will be saved in './Prep'

In [ ]:
# Import the necessary modules
import os, glob, warnings
from dataclasses import dataclass
import multiprocessing as mp
import numpy as np
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from scipy import ndimage
from scipy.interpolate import RectBivariateSpline
import matplotlib.pyplot as plt # For visualisation

# Ignore some nuisance warnings
warnings.filterwarnings("ignore", message="All-NaN slice")
warnings.filterwarnings("ignore", message="Mean of empty slice")


In [ ]:
INPUT_FILES = sorted(glob.glob("./Raw/jw*.fits")) #Input directory for rate files downloaded from MAST
OUTPUT_DIR  = "./Interim/"

"""
Constants for background and continuum subtraction - fine tune to your own needs!
"""

# Background subtraction constants
BKG_BOX       = 16      # Local background box size
N_BKG_ITER    = 3       # Number of background subtraction iterations
DO_ONE_OVER_F = True    # Remove 1/f noise

# Parameters for initial source detection
MASK_BOX      = 9       # Source detection size (pixels)
MASK_NSIGMA   = 2.5     # Source detection threshold
MASK_GROW     = 4       # Expand the source to avoid over-subtraction
HALO_BOX      = 8      # Scale for detecting extended halos / intracluster light (0 = off)
HALO_NSIGMA   = 4.0     # Threshold for halo detection

# Continuum subtraction parameters, default parameters are from Kashino+23
CONT_WIDTH    = 51      # Running-median kernel length along dispersion (pixels)
CONT_HOLE     = 9       # central hole excluded from the kernel (pixels)

# Extra masking for end of trace (generally leave these as they are)
END_WIDTH     = 21      
END_HOLE      = 9       
END_NSIGMA    = 5.0     
END_FRAC      = 0.3     
END_STEP      = 5       
END_RAMP      = 11


# Second pass line detection
LINE_NSIGMA   = 3.0     # emission-line mask threshold for the second pass (smoothed S/N)
LINE_MIN_PIX  = 6       # smallest line region kept (pixels)
LINE_GROW     = 2       # grow the line mask by this many pixels 

In [ ]:
# Filter ranges
LW_FILTERS = {
    "F250M": (2.41, 2.60), "F277W": (2.42, 3.13), "F300M": (2.83, 3.16),
    "F322W2": (2.43, 4.01), "F335M": (3.18, 3.53), "F356W": (3.14, 3.98),
    "F360M": (3.44, 3.80), "F410M": (3.86, 4.30), "F430M": (4.17, 4.40),
    "F444W": (3.88, 4.99), "F460M": (4.51, 4.74), "F480M": (4.66, 4.97),
}

# Approximate dispersion
DISPERSION_NM_PER_PIX = {"GRISMR": 1.00, "GRISMC": 0.98}

# Retrieve information about a WFSS file from its header
def wfss_setup(header):
    filt, pupil = header["FILTER"].strip(), header["PUPIL"].strip()
    if header.get("EXP_TYPE", "NRC_WFSS").strip() != "NRC_WFSS":
        raise ValueError(f"EXP_TYPE={header['EXP_TYPE']} is not NRC_WFSS")
    if pupil not in DISPERSION_NM_PER_PIX:
        raise ValueError(f"PUPIL={pupil}: expected GRISMR or GRISMC")
    if filt not in LW_FILTERS:
        raise ValueError(f"FILTER={filt} not in LW grism filter table")
    lo, hi = LW_FILTERS[filt]
    trace_len = int((hi - lo) * 1000 / DISPERSION_NM_PER_PIX[pupil])
    return {
        "filter": filt,
        "pupil": pupil,
        "disp_axis": 1 if pupil == "GRISMR" else 0,   
        "trace_len": trace_len,                     
    }

# Get the dispersion size and direction for a given WFSS image based on GRISM and FILTER
def oriented_box(cfg, along, cross):
    return (cross, along) if cfg["disp_axis"] == 1 else (along, cross)

In [ ]:
DO_NOT_USE = 1

# Load rate files
def load_rate(fn):
    print('Loading file: ',fn)
    with fits.open(fn) as hdul:
        hdr = hdul[0].header.copy()
        sci = hdul["SCI"].data.astype(float)
        err = hdul["ERR"].data.astype(float)
        dq = hdul["DQ"].data.astype(np.uint32)
        bad = ((dq & DO_NOT_USE) > 0) | ~np.isfinite(sci) | ~np.isfinite(err) | (err <= 0)
        bad[:4, :] = bad[-4:, :] = bad[:, :4] = bad[:, -4:] = True      # reference pixels
        return hdr, sci, err, dq, bad


# Write rate files
def write_products(fn_in, out_root, bkgsub, cont, contsub, src_mask, cfg, line_mask, end_weight):
    with fits.open(fn_in) as hdul:
        hdul["SCI"].data = bkgsub.astype(np.float32)
        hdul.writeto(f"{out_root}_bkgsub_rate.fits", overwrite=True)
        prim = hdul[0].copy()
        err = hdul["ERR"].copy()
    fits.HDUList([
        prim,
        fits.ImageHDU(contsub.astype(np.float32), name="SCI"),
        err,
        fits.ImageHDU(cont.astype(np.float32), name="CONT"),
        fits.ImageHDU(src_mask.astype(np.uint8), name="SRCMASK"),
        fits.ImageHDU(line_mask.astype(np.uint8), name="LINEMASK"),
        fits.ImageHDU(end_weight.astype(np.float32), name="ENDWEIGHT"),
    ]).writeto(f"{out_root}_contsub.fits", overwrite=True)

In [ ]:
# Local background subtraction
def local_background(img, mask, box=BKG_BOX, min_frac=0.3):
    ny, nx = img.shape
    by, bx = int(np.ceil(ny / box)), int(np.ceil(nx / box))
    pad = np.full((by * box, bx * box), np.nan)
    pad[:ny, :nx] = np.where(mask, np.nan, img)
    blocks = pad.reshape(by, box, bx, box).transpose(0, 2, 1, 3).reshape(by, bx, -1)
    coarse = np.nanmedian(blocks, axis=-1)
    coarse[np.isfinite(blocks).mean(axis=-1) < min_frac] = np.nan
    if np.all(np.isnan(coarse)):
        return np.zeros_like(img)
    idx = ndimage.distance_transform_edt(np.isnan(coarse), return_distances=False, return_indices=True)
    coarse = ndimage.median_filter(coarse[tuple(idx)], size=3, mode="nearest")
    yc, xc = (np.arange(by) + 0.5) * box - 0.5, (np.arange(bx) + 0.5) * box - 0.5
    spl = RectBivariateSpline(yc, xc, coarse, kx=min(3, by - 1), ky=min(3, bx - 1))
    return spl(np.clip(np.arange(ny), yc[0], yc[-1]), np.clip(np.arange(nx), xc[0], xc[-1]))

# Detect sources
def detect(img, excl, box, nsigma, sig, npix_min=1):
    use = ~excl & np.isfinite(img)
    num = ndimage.uniform_filter(np.where(use, img, 0.0), box, mode="constant")
    frac = ndimage.uniform_filter(use.astype(float), box, mode="constant")
    snr = (num / np.maximum(frac, 1e-3)) / (sig / np.sqrt(np.maximum(frac * box**2, 1)))
    det = (snr > nsigma) & (frac > 0.2)
    if npix_min > 1:                       # drop isolated specks
        lab, n = ndimage.label(det)
        if n:
            sizes = ndimage.sum(det, lab, np.arange(1, n + 1))
            det = np.isin(lab, np.flatnonzero(sizes >= npix_min) + 1)
    return det

# Build the source mask
def make_source_mask(img, bad):
    hp = img - local_background(img, bad)
    _, _, sig = sigma_clipped_stats(hp[~bad], sigma=3, maxiters=5)
    mask = detect(hp, bad, MASK_BOX, MASK_NSIGMA, sig, npix_min=MASK_BOX**2)
    mask = ndimage.binary_dilation(mask, iterations=MASK_GROW)
    if HALO_BOX:
        excl = bad | mask
        wide = local_background(img, excl, box=4 * HALO_BOX)
        halo = detect(img - wide, excl, HALO_BOX, HALO_NSIGMA, sig)
        mask |= ndimage.binary_dilation(halo, iterations=HALO_BOX // 4)
    return mask & ~bad

# Remove 1/f noise
def remove_row_stripes(img, mask, namp=4, min_good=0.25):
    ny, nx = img.shape
    d = np.where(mask, np.nan, img)
    full = np.nanmedian(d, axis=1)
    full = np.where(np.isfinite(full), full, 0.0)
    out = img.copy()
    edges = np.linspace(0, nx, namp + 1).astype(int) if nx == 2048 else np.array([0, nx])
    for x0, x1 in zip(edges[:-1], edges[1:]):
        sub = d[:, x0:x1]
        med = np.nanmedian(sub, axis=1)
        ok = (np.isfinite(sub).mean(axis=1) >= min_good) & np.isfinite(med)
        out[:, x0:x1] -= np.where(ok, med, full)[:, None]
    return out

# Combine all the functions above into one convenience function - can be run multiple times
def subtract_background(sci, bad, cfg, n_iter=N_BKG_ITER, one_over_f=DO_ONE_OVER_F):
    mask = make_source_mask(sci, bad)
    bk = sci.copy()
    for it in range(n_iter):
        excl = mask | bad
        bk = sci - local_background(sci, excl)
        if one_over_f:
            bk = remove_row_stripes(bk, excl)
        mask = make_source_mask(bk, bad)              
        print(f"  iter {it+1}: masked={mask.mean():.1%}")
    return np.where(bad, np.nan, bk), mask

In [ ]:
# Convenience function to define a running median with a hole
def holed_median(img, width, hole, axis, chunk=64):
    a = img if axis == 1 else img.T
    ny, nx = a.shape
    half, hh = width // 2, hole // 2
    offs = [d for d in range(-half, half + 1) if abs(d) > hh]
    pad = np.pad(a, ((0, 0), (half, half)), constant_values=np.nan)
    med, cnt = np.empty_like(a), np.empty(a.shape, int)
    for y0 in range(0, ny, chunk):
        blk = pad[y0:y0 + chunk]
        win = np.stack([blk[:, half + d: half + d + nx] for d in offs], axis=-1)
        med[y0:y0 + chunk] = np.nanmedian(win, axis=-1)  
        cnt[y0:y0 + chunk] = np.isfinite(win).sum(axis=-1)
    return (med, cnt) if axis == 1 else (med.T, cnt.T)

# Identify the transition from continuum to background in bright sources
def end_weights(cont, err, bad, cfg):
    axis, s = cfg["disp_axis"], END_STEP
    grad = np.full_like(cont, np.nan)
    level = np.full_like(cont, np.nan)
    if axis == 1:
        grad[:, s:-s] = cont[:, 2 * s:] - cont[:, :-2 * s]
        level[:, s:-s] = np.maximum(np.abs(cont[:, 2 * s:]), np.abs(cont[:, :-2 * s]))
    else:
        grad[s:-s] = cont[2 * s:] - cont[:-2 * s]
        level[s:-s] = np.maximum(np.abs(cont[2 * s:]), np.abs(cont[:-2 * s]))
    sig = np.sqrt(2) * 1.2533 * err / np.sqrt(CONT_WIDTH - CONT_HOLE)
    ends = (np.abs(grad) > END_NSIGMA * sig) & (np.abs(grad) > END_FRAC * level) & ~bad & np.isfinite(grad)
    reach = np.ones(CONT_WIDTH, bool)
    ends = ndimage.binary_dilation(ends, structure=reach[None, :] if axis == 1 else reach[:, None])
    return ndimage.uniform_filter1d(ends.astype(float), END_RAMP, axis=axis)


def find_lines(contsub, err, bad):
    snr = np.where(bad | ~np.isfinite(contsub), 0.0, contsub / err)
    snr = ndimage.uniform_filter(snr, size=3, mode="constant")
    v = snr[~bad]
    snr = (snr - np.median(v)) / (1.4826 * np.median(np.abs(v - np.median(v))))
    lines = (snr > LINE_NSIGMA) & ~bad
    lab, n = ndimage.label(lines)
    if n:
        sizes = ndimage.sum(lines, lab, np.arange(1, n + 1))
        lines = np.isin(lab, np.flatnonzero(sizes >= LINE_MIN_PIX) + 1)
    return ndimage.binary_dilation(lines, iterations=LINE_GROW) & ~bad

def continuum_subtract(img, err, bad, cfg):
    axis = cfg["disp_axis"]
    work = np.where(bad, np.nan, img)
    # Pass 1: Long+short kernals blended towards trace ends
    c_long, _ = holed_median(work, CONT_WIDTH, CONT_HOLE, axis)
    c_short, _ = holed_median(work, END_WIDTH, END_HOLE, axis)
    w = end_weights(c_long, err, bad, cfg)
    cont1 = w * c_short + (1 - w) * c_long
    # Pass 2: As above with emission lines masked
    line_mask = find_lines(img - cont1, err, bad)
    masked = np.where(line_mask, np.nan, work)
    c_long2, n_long = holed_median(masked, CONT_WIDTH, CONT_HOLE, axis)
    c_short2, n_short = holed_median(masked, END_WIDTH, END_HOLE, axis)
    c_long2 = np.where(n_long >= (CONT_WIDTH - CONT_HOLE) // 2, c_long2, c_long)    
    c_short2 = np.where(n_short >= (END_WIDTH - END_HOLE) // 2, c_short2, c_short) 
    cont = w * c_short2 + (1 - w) * c_long2
    cont = np.where(np.isfinite(cont), cont, 0.0)
    contsub = np.where(bad, np.nan, img - cont)
    return contsub, cont, line_mask, w

In [ ]:
results = {}

# Load all the headers, required for the reduction
def inputs():
    for fn in INPUT_FILES:
        hdr, sci, err, dq, bad = load_rate(fn)
        if 'GRISM' not in hdr['PUPIL']:
            print('Non-WFSS file loaded, skipping')
            continue
        yield fn, hdr, sci, err, bad, None
"""
If for some reason the generator above doesn't work, the two lines below can be used instead, 
but is much slower, especially with large programs.
"""
# files = [(fn,) + tuple(load_rate(fn)[i] for i in (0, 1, 2, 4)) for fn in INPUT_FILES]
# for name, hdr, sci, err, bad in files:

for name, hdr, sci, err, bad, true_sky in inputs():    
    cfg = wfss_setup(hdr)
    print(f"{os.path.basename(name)}  {cfg['filter']}/{cfg['pupil']}")
    bkg, smask = subtract_background(sci, bad, cfg)
    cs, cont, lmask, endw = continuum_subtract(bkg, err, bad, cfg)
    root = os.path.join(OUTPUT_DIR, os.path.basename(name).replace("_rate.fits", ""))
    write_products(name, root, bkg, cont, cs, smask, cfg, lmask, endw)
    results[name] = dict(cfg=cfg, sci=sci, bkg=bkg, smask=smask, cs=cs, cont=cont, err=err, bad=bad,
                         lmask=lmask, endw=endw)
    break